# Building a rating algorithm with `open-rating-engine`

This notebook builds a motor insurance rating algorithm from scratch with the `open_rating_engine` pricing pipeline, then scores two example policies through it.

An algorithm is a set of **layers**. Each layer computes one column (or reshapes a table), and declares which other layers it depends on. The pipeline sorts the layers into a dependency graph (DAG) and runs them in order, running independent layers in parallel.

| Stage | What it does |
|---|---|
| Data prep | Brings policy attributes down onto each vehicle, bands driver age and customer tenure, flags hire car cover |
| Exposure rating | An **ONNX** model predicts the claim cost of each vehicle |
| Claims rating | Experience-rates the policy from its own 5-year claims history, with an IBNR uplift |
| Statistical premium | Credibility-blends exposure and claims rating, then adds expense and profit loadings |
| Demand premium | Applies rating-table multipliers |
| Cap / cup | Limits how far a renewal premium can move from last year's |
| Final premium | The offered gross written premium (GWP) |
| Demand model | An H2O GBM, converted to **ONNX**, predicts the probability the customer accepts the offered premium |
| Reporting | Offered and expected (conversion-weighted) KPIs |

**Prerequisites**

- Install the package with the ONNX and plotting extras. From the `open-rating-engine` folder: `pip install -e ".[onnx,viz]"`
- Run the notebook from the `.examples` folder, so the `models/` folder resolves.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from mlflow.pyfunc import PythonModelContext

from open_rating_engine import (
    ColumnCalculation,
    OnnxScoring,
    PyfuncPricingPipeline,
    RatingTablesLayer,
    TableJoin,
)

## 1. Input data

The pipeline takes a dict of DataFrames, one per table. Every column is prefixed with its table name, so a column can be referenced unambiguously anywhere in the algorithm.

The example request holds two policies:

- `149110b481`: a **renewal** with 3 vehicles, two of which have hire car cover
- `0a01d5c4c2`: **new business** with 2 vehicles and no previous premium

In [ ]:
DATA_DIR = Path("data").resolve() / "origination"

# Load or fetch data here
risk            = pd.read_parquet(f"{DATA_DIR}\\risk.parquet")
cover           = pd.read_parquet(f"{DATA_DIR}\\cover.parquet")
policy          = pd.read_parquet(f"{DATA_DIR}\\policy.parquet")
claims_history  = pd.read_parquet(f"{DATA_DIR}\\claims_history.parquet")


data = {
    "policy": policy,
    "risk": risk,
    "cover": cover,
    "claims_history": claims_history,
}

## 2. Data structure

The tables form a hierarchy: a policy has many vehicles (`risk`) and many years of `claims_history`, and each vehicle has many `cover` rows.

The `aggregate()` helper used inside `ColumnCalculation` expressions moves values up and down this hierarchy, such as summing vehicle claim costs onto the policy or copying the policy's transaction type onto each vehicle. It reads the join keys from the data structure, so expressions never repeat them.

In [ ]:
data_structure = [
    {"name": "policy", "parent": None, "parent_key": None, "child_key": None, "cardinality": None},
    {
        "name": "risk",
        "parent": "policy",
        "parent_key": "policy.policy_number",
        "child_key": "risk.policy_number",
        "cardinality": "one_to_many",
    },
    {
        "name": "cover",
        "parent": "risk",
        "parent_key": "risk.risk_number",
        "child_key": "cover.risk_number",
        "cardinality": "one_to_many",
    },
    {
        "name": "claims_history",
        "parent": "policy",
        "parent_key": "policy.policy_number",
        "child_key": "claims_history.policy_number",
        "cardinality": "one_to_many",
    },
]

## 3. Catalog

The catalog holds static lookup tables (assumptions, reference data) shared by every layer. Layers reach them two ways:

- inside an expression, as `catalog['<name>']`
- through a `TableJoin` with `right_source="catalog"`

Because the catalog is separate from the layers, an assumption such as an expense loading can be updated without touching the algorithm.

In [ ]:
catalog = {
    # Split of predicted claim cost by cover type.
    "rating_table_pred_claim_cost": pd.DataFrame({
        "cover_type": ["core", "windscreen", "hire_car"],
        "pred_claim_cost_split": [0.7, 0.1, 0.2],
    }),
    # IBNR uplift by claims-history year: recent years are still developing.
    "assumptions_ibnr": pd.DataFrame({
        "term": [-1, -2, -3, -4, -5, -6, -7],
        "ibnr_factor": [1.3, 1.1, 1.0, 1.0, 1.0, 1.0, 1.0],
    }),
    # Expense and profit loadings: fixed amounts in dollars, variable amounts as a share of risk cost.
    "assumptions_expenses": pd.DataFrame({
        "expense_type": [
            "expense_administration_fixed",
            "expense_administration_variable",
            "expense_claims_handling_fixed",
            "expense_claims_handling_variable",
            "expense_commission_fixed",
            "expense_commission_variable",
            "expense_reinsurance_expense",
            "expense_investment_income",
            "expense_target_profit_margin_percent",
        ],
        "value": [75.00, 0.02, 0.00, 0.05, 0.00, 0.05, 0.00, 0.05, 0.10],
    }),
    # Maximum premium increase (cap) and decrease (cup) at renewal, by customer tenure.
    "assumptions_cap_cup": pd.DataFrame({
        "customer_tenure_band": ["0-5", "6-10", "10+"],
        "cap_percentage": [0.5, 0.2, 0.1],
        "cup_percentage": [-0.05, -0.10, -0.20],
    }),
}

for name, table in catalog.items():
    print(name)
    display(table)

## 4. Model artifacts

Model-scoring layers don't store file paths. Each one stores an **artifact key**, and the key resolves to a file through the MLflow `PythonModelContext`. That keeps the algorithm portable: the same layers score against local files here and against downloaded registry files in production.

Both models are ONNX files scored with `onnxruntime`, so no JVM is needed. The demand model is the H2O MOJO in `models/motor_demand_onnx/model.zip`, converted with `onnxmltools.convert_h2o` and reshaped to one input per feature, with the positive-class probability as its output.

In [ ]:
MODELS_DIR = Path("models").resolve()

artifacts = {
    "motor_risk_cost_onnx": MODELS_DIR / "motor_risk_cost_onnx" / "pipeline_onnx.onnx",
    "motor_demand_onnx": MODELS_DIR / "motor_demand_onnx" / "h2o_gbm.onnx",
}
missing = [key for key, path in artifacts.items() if not path.exists()]
assert not missing, f"Model files not found for {missing}. Run this notebook from open-rating-engine/.examples."

context = PythonModelContext(
    artifacts={key: str(path) for key, path in artifacts.items()},
    model_config={
        # "local" makes predict() take and return DataFrames.
        "environment": "local",
    },
)

## 5. Layers

Each section below builds a list of layers for one stage of the algorithm. Every layer has:

- `name`: a unique identifier. Here it matches the column the layer writes.
- `dependencies`: names of the layers that must run first. The pipeline does not read expressions to work this out, so **every column produced by another layer must be listed**. Columns supplied in the input data need no dependency.

`ColumnCalculation` expressions are Python strings evaluated with these names in scope:

| Name | Meaning |
|---|---|
| `df` | The layer's target table |
| `data` | Every table, keyed by name |
| `catalog` | The catalog tables |
| `np`, `pd` | NumPy and pandas |
| `aggregate(source, agg_func, source_column)` | Aggregates `source` onto the target table using the data structure keys |

### 5.1 Data prep

Copies policy attributes onto each vehicle, bands driver age and customer tenure, and flags vehicles with hire car cover.

In [ ]:
data_prep = [
    ColumnCalculation(
        name="risk.transaction_type",
        target_table="risk",
        output_column="risk.transaction_type",
        calculation="aggregate(source='policy', source_column='policy.transaction_type', agg_func='first')",
        description="Policy transaction type (NB / RN) copied onto each vehicle.",
    ),
    ColumnCalculation(
        name="risk.customer_tenure",
        target_table="risk",
        output_column="risk.customer_tenure",
        calculation="aggregate(source='policy', source_column='policy.customer_tenure', agg_func='first')",
        description="Customer tenure copied onto each vehicle.",
    ),
    ColumnCalculation(
        name="risk.customer_tenure_band",
        target_table="risk",
        output_column="risk.customer_tenure_band",
        calculation="pd.cut(df['risk.customer_tenure'], bins=[-1, 5, 10, np.inf], labels=['0-5', '6-10', '10+'])",
        dependencies=["risk.customer_tenure"],
        description="Customer tenure band.",
    ),
    ColumnCalculation(
        name="risk.risk_age_band",
        target_table="risk",
        output_column="risk.risk_age_band",
        calculation=(
            "pd.cut(df['risk.driver_youngest_age'], bins=list(range(15, 66, 10)) + [np.inf], "
            "labels=['15-24', '25-34', '35-44', '45-54', '55-64', '65+'])"
        ),
        description="Youngest driver age band.",
    ),
    ColumnCalculation(
        name="cover.cover_flag_hire_car",
        target_table="cover",
        output_column="cover.cover_flag_hire_car",
        calculation="np.where(df['cover.cover_type'] == 'hire_car', 1, 0)",
        description="Flags hire car cover rows.",
    ),
    ColumnCalculation(
        name="risk.cover_flag_hire_car",
        target_table="risk",
        output_column="risk.cover_flag_hire_car",
        calculation="aggregate(source='cover', source_column='cover.cover_flag_hire_car', agg_func='max')",
        dependencies=["cover.cover_flag_hire_car"],
        description="1 if the vehicle has hire car cover.",
    ),
]

### 5.2 Exposure rating

An ONNX model predicts each vehicle's claim cost from its rating factors. `column_map` maps the ONNX graph's input names (keys) to columns on the target table (values), so the model can be trained on different column names from the ones in the live data. Each input is cast to the dtype the graph declares.

The vehicle predictions are then summed onto the policy.

In [ ]:
exposure_rating = [
    OnnxScoring(
        name="risk.pred_claim_cost",
        target_table="risk",
        output_column="risk.pred_claim_cost",
        model_artifact_key="motor_risk_cost_onnx",
        column_map={
            "driver_youngest_age": "risk.driver_youngest_age",
            "vehicle_market_value": "risk.vehicle_market_value",
            "risk_usage": "risk.vehicle_usage",
            "transaction_type": "risk.transaction_type",
            "claims_history_claim_count_5_years": "risk.claims_history_claim_count_5_years",
            "coverage_hire_car": "risk.cover_flag_hire_car",
            "driver_youngest_occupation": "risk.driver_youngest_occupation",
            "location_geoid": "risk.location_geoid",
            "location_population": "risk.location_population",
            "location_proportion_white": "risk.location_proportion_white",
            "location_statefp": "risk.location_statefp",
            "risk_average_milage": "risk.average_mileage",
            "risk_basic_excess": "risk.basic_excess",
            "risk_customer_tenure": "risk.customer_tenure",
            "vehicle_age": "risk.vehicle_age",
            "vehicle_fuel_type": "risk.vehicle_fuel_type",
            "vehicle_make": "risk.vehicle_make",
        },
        dependencies=["risk.transaction_type", "risk.cover_flag_hire_car", "risk.customer_tenure"],
        description="Claim cost scored from an ONNX model.",
    ),
    ColumnCalculation(
        name="policy.pred_claim_cost",
        target_table="policy",
        output_column="policy.pred_claim_cost",
        calculation="aggregate(source='risk', source_column='risk.pred_claim_cost', agg_func='sum')",
        dependencies=["risk.pred_claim_cost"],
        description="Exposure-rated claim cost for the policy.",
    ),
]

### 5.3 Claims rating

Rates the policy on its own claims experience. A `TableJoin` brings the IBNR factor from the catalog onto each claims-history year, so recent years still developing are uplifted to their expected ultimate cost. The ultimate cost per vehicle-year is then applied to the policy's current vehicle count.

In [ ]:
claims_rating = [
    ColumnCalculation(
        name="policy.vehicle_count",
        target_table="policy",
        output_column="policy.vehicle_count",
        calculation="aggregate(source='risk', source_column='risk.risk_number', agg_func='count')",
        description="Number of vehicles on the policy.",
    ),
    TableJoin(
        name="claims_history_ibnr",
        left_table="claims_history",
        right_table="assumptions_ibnr",
        right_source="catalog",
        left_on="claims_history.term",
        right_on="term",
        columns=["ibnr_factor"],
        how="left",
        description="Join IBNR factor onto claims_history.",
    ),
    ColumnCalculation(
        name="claims_history.incurred_ultimate",
        target_table="claims_history",
        output_column="claims_history.incurred_ultimate",
        calculation="df['claims_history.incurred'] * df['ibnr_factor']",
        dependencies=["claims_history_ibnr"],
        description="Ultimate incurred claims.",
    ),
    ColumnCalculation(
        name="claims_history.cost_per_vehicle_ultimate",
        target_table="claims_history",
        output_column="claims_history.cost_per_vehicle_ultimate",
        calculation="df['claims_history.incurred_ultimate'] / df['claims_history.average_number_vehicles']",
        dependencies=["claims_history.incurred_ultimate"],
        description="Ultimate incurred claims per vehicle, by year.",
    ),
    ColumnCalculation(
        name="policy.incurred_ultimate",
        target_table="policy",
        output_column="policy.incurred_ultimate",
        calculation="aggregate(source='claims_history', source_column='claims_history.incurred_ultimate', agg_func='sum')",
        dependencies=["claims_history.incurred_ultimate"],
    ),
    ColumnCalculation(
        name="policy.average_number_vehicles",
        target_table="policy",
        output_column="policy.average_number_vehicles",
        calculation="aggregate(source='claims_history', source_column='claims_history.average_number_vehicles', agg_func='sum')",
        description="Vehicle-years of exposure in the claims history.",
    ),
    ColumnCalculation(
        name="policy.claim_count",
        target_table="policy",
        output_column="policy.claim_count",
        calculation="aggregate(source='claims_history', source_column='claims_history.claim_count', agg_func='sum')",
    ),
    ColumnCalculation(
        name="policy.cost_per_vehicle",
        target_table="policy",
        output_column="policy.cost_per_vehicle",
        calculation="df['policy.incurred_ultimate'] / df['policy.average_number_vehicles']",
        dependencies=["policy.incurred_ultimate", "policy.average_number_vehicles"],
        description="Claims-history cost per vehicle-year.",
    ),
    ColumnCalculation(
        name="policy.claims_rated_pred_claim_cost",
        target_table="policy",
        output_column="policy.claims_rated_pred_claim_cost",
        calculation="df['policy.cost_per_vehicle'] * df['policy.vehicle_count']",
        dependencies=["policy.cost_per_vehicle", "policy.vehicle_count"],
        description="Claims-rated claim cost for the policy.",
    ),
]

### 5.4 Statistical premium

Blends the two claim cost estimates with a credibility weight `Z = n / (n + 100)`, where `n` is the policy's claim count, so policies with more claims history lean more on their own experience. The blended policy cost is allocated back to each vehicle in proportion to its exposure-rated cost, and the expense and profit loadings from the catalog are added.

The `expense()` helper only builds expression text. The layers still hold plain strings, so the algorithm stays serialisable.

In [ ]:
def expense(expense_type: str) -> str:
    """Expression text that looks up one value from the assumptions_expenses catalog table."""
    return f"catalog['assumptions_expenses'].set_index('expense_type').loc['{expense_type}', 'value']"


RISK_COST = "df['risk.premium_stat_prem_risk_cost']"

statistical_premium = [
    ColumnCalculation(
        name="policy.credibility_weighting",
        target_table="policy",
        output_column="policy.credibility_weighting",
        calculation="df['policy.claim_count'] / (df['policy.claim_count'] + 100)",
        dependencies=["policy.claim_count"],
        description="Policy credibility weighting.",
    ),
    ColumnCalculation(
        name="policy.premium_stat_prem_risk_cost",
        target_table="policy",
        output_column="policy.premium_stat_prem_risk_cost",
        calculation=(
            "df['policy.credibility_weighting'] * df['policy.claims_rated_pred_claim_cost']"
            " + (1 - df['policy.credibility_weighting']) * df['policy.pred_claim_cost']"
        ),
        dependencies=[
            "policy.credibility_weighting",
            "policy.claims_rated_pred_claim_cost",
            "policy.pred_claim_cost",
        ],
        description="Credibility-weighted policy risk cost.",
    ),
    ColumnCalculation(
        name="risk.policy_pred_claim_cost",
        target_table="risk",
        output_column="risk.policy_pred_claim_cost",
        calculation="aggregate(source='policy', source_column='policy.pred_claim_cost', agg_func='sum')",
        dependencies=["policy.pred_claim_cost"],
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_risk_cost",
        target_table="risk",
        output_column="risk.premium_stat_prem_risk_cost",
        calculation=(
            "aggregate(source='policy', source_column='policy.premium_stat_prem_risk_cost', agg_func='sum')"
            " * df['risk.pred_claim_cost'] / df['risk.policy_pred_claim_cost']"
        ),
        dependencies=["policy.premium_stat_prem_risk_cost", "risk.pred_claim_cost", "risk.policy_pred_claim_cost"],
        description="Policy risk cost allocated to each vehicle.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_administration_expense",
        target_table="risk",
        output_column="risk.premium_stat_prem_administration_expense",
        calculation=f"{expense('expense_administration_fixed')} + {expense('expense_administration_variable')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Administration expense component.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_claims_handling_expense",
        target_table="risk",
        output_column="risk.premium_stat_prem_claims_handling_expense",
        calculation=f"{expense('expense_claims_handling_fixed')} + {expense('expense_claims_handling_variable')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Claims handling expense component.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_commission_expense",
        target_table="risk",
        output_column="risk.premium_stat_prem_commission_expense",
        calculation=f"{expense('expense_commission_fixed')} + {expense('expense_commission_variable')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Commission expense component.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_reinsurance_expense",
        target_table="risk",
        output_column="risk.premium_stat_prem_reinsurance_expense",
        calculation=f"{expense('expense_reinsurance_expense')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Reinsurance expense component.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_investment_income",
        target_table="risk",
        output_column="risk.premium_stat_prem_investment_income",
        calculation=f"-{expense('expense_investment_income')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Investment income credit.",
    ),
    ColumnCalculation(
        name="risk.premium_stat_prem_target_profit_margin",
        target_table="risk",
        output_column="risk.premium_stat_prem_target_profit_margin",
        calculation=f"{expense('expense_target_profit_margin_percent')} * {RISK_COST}",
        dependencies=["risk.premium_stat_prem_risk_cost"],
        description="Target profit margin component.",
    ),
]

stat_prem_components = [
    "risk.premium_stat_prem_risk_cost",
    "risk.premium_stat_prem_administration_expense",
    "risk.premium_stat_prem_claims_handling_expense",
    "risk.premium_stat_prem_commission_expense",
    "risk.premium_stat_prem_reinsurance_expense",
    "risk.premium_stat_prem_investment_income",
    "risk.premium_stat_prem_target_profit_margin",
]

statistical_premium.append(
    ColumnCalculation(
        name="risk.premium_stat_prem_total",
        target_table="risk",
        output_column="risk.premium_stat_prem_total",
        calculation=" + ".join(f"df['{column}']" for column in stat_prem_components),
        dependencies=stat_prem_components,
        description="Total statistical (technical) premium.",
    )
)

### 5.5 Demand premium

A `RatingTablesLayer` looks up a factor from each of its rating tables and combines them with `formula` (by default, it multiplies them). The key column of each rating table must match a column on the target table. A row with no match takes the table's `default_values` entry.

All factors are 1.0 here, so the demand premium equals the statistical premium. They exist as the levers a pricing team adjusts, or that the price optimisers in `open_rating_engine.optimization` tune.

In [ ]:
demand_premium = [
    RatingTablesLayer(
        name="risk.demand_multiplier",
        target_table="risk",
        output_column="risk.demand_multiplier",
        rating_tables={
            "risk_age_band": pd.DataFrame({
                "risk.risk_age_band": ["15-24", "25-34", "35-44", "45-54", "55-64", "65+"],
                "value": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
            }),
            "customer_tenure_band": pd.DataFrame({
                "risk.customer_tenure_band": ["0-5", "6-10", "10+"],
                "value": [1.0, 1.0, 1.0],
            }),
        },
        formula="risk_age_band['value'] * customer_tenure_band['value']",
        default_values={"risk_age_band": 1, "customer_tenure_band": 1},
        dependencies=["risk.risk_age_band", "risk.customer_tenure_band"],
        description="Demand multiplier from rating tables.",
    ),
    RatingTablesLayer(
        name="risk.demand_multiplier_individual",
        target_table="risk",
        output_column="risk.demand_multiplier_individual",
        # Placeholder for per-vehicle multipliers: unmatched vehicles take the default of 1.
        rating_tables={
            "risk_number": pd.DataFrame({"risk.risk_number": ["temp"], "value": [1.0]}),
        },
        formula="risk_number['value']",
        default_values={"risk_number": 1},
        description="Individual (per-vehicle) demand multiplier.",
    ),
    ColumnCalculation(
        name="risk.premium_demand_total",
        target_table="risk",
        output_column="risk.premium_demand_total",
        calculation="df['risk.premium_stat_prem_total'] * df['risk.demand_multiplier'] * df['risk.demand_multiplier_individual']",
        dependencies=["risk.premium_stat_prem_total", "risk.demand_multiplier", "risk.demand_multiplier_individual"],
        description="Demand-adjusted premium.",
    ),
]

### 5.6 Cap / cup

Limits the year-on-year premium movement for renewals. The cap (maximum increase) and cup (maximum decrease) come from the catalog by tenure band. New business has no previous premium, so it passes through uncapped.

In [ ]:
cap_cup_premium = [
    TableJoin(
        name="policy_assumptions_cap_cup",
        left_table="risk",
        right_table="assumptions_cap_cup",
        right_source="catalog",
        left_on="risk.customer_tenure_band",
        right_on="customer_tenure_band",
        columns=["cap_percentage", "cup_percentage"],
        how="left",
        dependencies=["risk.customer_tenure_band"],
        description="Bring cap/cup percentages onto risk.",
    ),
    ColumnCalculation(
        name="cap_percentage_filled",
        target_table="risk",
        output_column="cap_percentage_filled",
        calculation="df['cap_percentage'].fillna(2)",
        dependencies=["policy_assumptions_cap_cup"],
        description="Fill missing cap percentage.",
    ),
    ColumnCalculation(
        name="cup_percentage_filled",
        target_table="risk",
        output_column="cup_percentage_filled",
        calculation="df['cup_percentage'].fillna(-0.5)",
        dependencies=["policy_assumptions_cap_cup"],
        description="Fill missing cup percentage.",
    ),
    ColumnCalculation(
        name="risk.premium_comm_prem_cap_cup",
        target_table="risk",
        output_column="risk.premium_comm_prem_cap_cup",
        calculation=(
            "np.where("
            "df['risk.previous_gwp'].isna(), "
            "df['risk.premium_demand_total'], "
            "np.maximum("
            "np.minimum(df['risk.premium_demand_total'], df['risk.previous_gwp'] * (1 + df['cap_percentage_filled'])), "
            "df['risk.previous_gwp'] * (1 + df['cup_percentage_filled'])"
            "))"
        ),
        dependencies=["risk.premium_demand_total", "cap_percentage_filled", "cup_percentage_filled"],
        description="Cap/cup the demand premium; new business (null previous_gwp) passes through uncapped.",
    ),
    ColumnCalculation(
        name="risk.premium_final",
        target_table="risk",
        output_column="risk.premium_final",
        calculation="df['risk.premium_comm_prem_cap_cup']",
        dependencies=["risk.premium_comm_prem_cap_cup"],
        description="Final offered premium.",
    ),
]

### 5.7 Final premium

In [ ]:
final_premium = [
    ColumnCalculation(
        name="risk.gwp",
        target_table="risk",
        output_column="risk.gwp",
        calculation="df['risk.premium_comm_prem_cap_cup']",
        dependencies=["risk.premium_comm_prem_cap_cup"],
        description="Gross written premium.",
    ),
    ColumnCalculation(
        name="policy.gwp",
        target_table="policy",
        output_column="policy.gwp",
        calculation="aggregate(source='risk', source_column='risk.gwp', agg_func='sum')",
        dependencies=["risk.gwp"],
        description="Gross written premium for the policy.",
    ),
]

### 5.8 Demand model

An H2O GBM, converted to ONNX, predicts the probability that the customer accepts the offered premium. Its features describe the vehicle, its price position against competitors and last year's premium, and one-hot encodings of usage and transaction type. These are built as ordinary layers first, then passed to the model through `column_map` (H2O feature name, which is also the ONNX input name, to target table column).

The converted graph's only output is the probability of the positive class, so the layer writes that directly.

In [ ]:
demand_features = [
    ColumnCalculation(
        name="cat_onehot__risk_usage_Private",
        target_table="risk",
        output_column="cat_onehot__risk_usage_Private",
        calculation="(df['risk.vehicle_usage'] == 'Private').astype(int)",
        description="One-hot private vehicle usage.",
    ),
    ColumnCalculation(
        name="cat_onehot__risk_usage_Rideshare",
        target_table="risk",
        output_column="cat_onehot__risk_usage_Rideshare",
        calculation="(df['risk.vehicle_usage'] == 'Rideshare').astype(int)",
        description="One-hot rideshare vehicle usage.",
    ),
    ColumnCalculation(
        name="cat_onehot__transaction_type_RN",
        target_table="risk",
        output_column="cat_onehot__transaction_type_RN",
        calculation="(df['risk.transaction_type'] == 'RN').astype(int)",
        dependencies=["risk.transaction_type"],
        description="One-hot renewal transaction type.",
    ),
    ColumnCalculation(
        name="num__distance_to_competitor_percent",
        target_table="risk",
        output_column="num__distance_to_competitor_percent",
        calculation=(
            "np.where(df['risk.competitor_premiums_average'] != 0, "
            "(df['risk.gwp'] - df['risk.competitor_premiums_average']) / df['risk.competitor_premiums_average'], "
            "np.nan)"
        ),
        dependencies=["risk.gwp"],
        description="Relative distance to competitor premium.",
    ),
    ColumnCalculation(
        name="num__distance_to_competitor_abs",
        target_table="risk",
        output_column="num__distance_to_competitor_abs",
        calculation="np.abs(df['risk.gwp'] - df['risk.competitor_premiums_average'])",
        dependencies=["risk.gwp"],
        description="Absolute distance to competitor premium.",
    ),
    ColumnCalculation(
        name="num__price_change",
        target_table="risk",
        output_column="num__price_change",
        calculation=(
            "np.where(df['risk.previous_gwp'] != 0, "
            "(df['risk.gwp'] - df['risk.previous_gwp']) / df['risk.previous_gwp'], "
            "np.nan)"
        ),
        dependencies=["risk.gwp"],
        description="Relative price change vs previous premium.",
    ),
    ColumnCalculation(
        name="num__gwp",
        target_table="risk",
        output_column="num__gwp",
        calculation="df['risk.gwp']",
        dependencies=["risk.gwp"],
        description="Numeric GWP feature.",
    ),
]

demand_model = demand_features + [
    OnnxScoring(
        name="risk.conversion_rate",
        target_table="risk",
        output_column="risk.conversion_rate",
        model_artifact_key="motor_demand_onnx",
        column_map={
            "num__driver_youngest_age": "risk.driver_youngest_age",
            "num__vehicle_market_value": "risk.vehicle_market_value",
            "num__distance_to_competitor_percent": "num__distance_to_competitor_percent",
            "num__distance_to_competitor_abs": "num__distance_to_competitor_abs",
            "num__price_change": "num__price_change",
            "num__gwp": "num__gwp",
            "cat_onehot__risk_usage_Private": "cat_onehot__risk_usage_Private",
            "cat_onehot__risk_usage_Rideshare": "cat_onehot__risk_usage_Rideshare",
            "cat_onehot__transaction_type_RN": "cat_onehot__transaction_type_RN",
        },
        dependencies=[layer.name for layer in demand_features],
        description="Demand conversion rate scored from an H2O GBM converted to ONNX.",
    ),
]

### 5.9 Reporting

KPIs at the offered premium, and *expected* KPIs weighted by the conversion probability.

In [ ]:
reporting = [
    ColumnCalculation(
        name="risk.reporting_previous_gwp",
        target_table="risk",
        output_column="risk.reporting_previous_gwp",
        calculation="df['risk.previous_gwp']",
        description="Previous GWP.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_gwp",
        target_table="risk",
        output_column="risk.reporting_offered_gwp",
        calculation="df['risk.premium_final']",
        dependencies=["risk.premium_final"],
        description="Offered GWP.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_technical_premium",
        target_table="risk",
        output_column="risk.reporting_offered_technical_premium",
        calculation="df['risk.premium_stat_prem_total']",
        dependencies=["risk.premium_stat_prem_total"],
        description="Offered technical premium.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_profit",
        target_table="risk",
        output_column="risk.reporting_offered_profit",
        calculation="df['risk.reporting_offered_gwp'] - df['risk.reporting_offered_technical_premium']",
        dependencies=["risk.reporting_offered_gwp", "risk.reporting_offered_technical_premium"],
        description="Offered profit.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_loss_ratio",
        target_table="risk",
        output_column="risk.reporting_offered_loss_ratio",
        calculation="df['risk.reporting_offered_technical_premium'] / df['risk.reporting_offered_gwp']",
        dependencies=["risk.reporting_offered_technical_premium", "risk.reporting_offered_gwp"],
        description="Offered loss ratio.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_profit_margin",
        target_table="risk",
        output_column="risk.reporting_offered_profit_margin",
        calculation="df['risk.reporting_offered_profit'] / df['risk.reporting_offered_gwp']",
        dependencies=["risk.reporting_offered_profit", "risk.reporting_offered_gwp"],
        description="Offered profit margin.",
    ),
    ColumnCalculation(
        name="risk.reporting_offered_premium_change",
        target_table="risk",
        output_column="risk.reporting_offered_premium_change",
        calculation="df['risk.reporting_offered_gwp'] / df['risk.previous_gwp'] - 1",
        dependencies=["risk.reporting_offered_gwp"],
        description="Offered premium change.",
    ),
    ColumnCalculation(
        name="risk.reporting_expected_gwp",
        target_table="risk",
        output_column="risk.reporting_expected_gwp",
        calculation="df['risk.reporting_offered_gwp'] * df['risk.conversion_rate']",
        dependencies=["risk.reporting_offered_gwp", "risk.conversion_rate"],
        description="Expected GWP.",
    ),
    ColumnCalculation(
        name="risk.reporting_expected_technical_premium",
        target_table="risk",
        output_column="risk.reporting_expected_technical_premium",
        calculation="df['risk.reporting_offered_technical_premium'] * df['risk.conversion_rate']",
        dependencies=["risk.reporting_offered_technical_premium", "risk.conversion_rate"],
        description="Expected technical premium.",
    ),
    ColumnCalculation(
        name="risk.reporting_expected_profit",
        target_table="risk",
        output_column="risk.reporting_expected_profit",
        calculation="df['risk.reporting_offered_profit'] * df['risk.conversion_rate']",
        dependencies=["risk.reporting_offered_profit", "risk.conversion_rate"],
        description="Expected profit.",
    ),
    ColumnCalculation(
        name="risk.reporting_expected_profit_margin",
        target_table="risk",
        output_column="risk.reporting_expected_profit_margin",
        calculation="df['risk.reporting_expected_profit'] / df['risk.reporting_expected_gwp']",
        dependencies=["risk.reporting_expected_profit", "risk.reporting_expected_gwp"],
        description="Expected profit margin.",
    ),
]

## 6. Build the pipeline

Register the layers, the catalog and the data structure. The order layers are added in doesn't matter: the pipeline works out the run order from the dependencies, and raises an error if a dependency is missing or circular.

In [ ]:
layers = (
    data_prep
    + exposure_rating
    + claims_rating
    + statistical_premium
    + demand_premium
    + cap_cup_premium
    + final_premium
    + demand_model
    + reporting
)

pipeline = PyfuncPricingPipeline(context)

for layer in layers:
    pipeline.add_parameter(layer)

for name, table in catalog.items():
    pipeline.set_catalog(name, table)

pipeline.options["data_structure"] = data_structure

print(f"{len(pipeline.get_layer_names())} layers registered")

The execution plan groups the layers into **waves**. Every layer in a wave depends only on earlier waves, so the column calculations in a wave run in parallel.

In [ ]:
pipeline.print_execution_plan()

`plot_dag()` draws the same graph interactively. It needs the `viz` extra (`plotly`, `networkx`).

In [ ]:
pipeline.plot_dag(height=800)

## 7. Score

`load_context` resolves the artifact keys to files and loads the ONNX models. `predict` then runs every wave and returns the input tables with every computed column added.

In [ ]:
pipeline.load_context(context)

result = pipeline.predict(data)

### Premium build-up by vehicle

In [ ]:
build_up_columns = [
    "risk.policy_number",
    "risk.risk_number",
    "risk.transaction_type",
    "risk.pred_claim_cost",
    *stat_prem_components,
    "risk.premium_stat_prem_total",
    "risk.premium_demand_total",
    "risk.previous_gwp",
    "risk.premium_final",
    "risk.conversion_rate",
]

result["risk"][build_up_columns].set_index("risk.risk_number").T.round(2)

### Policy summary

The renewal's offered premium is capped relative to last year's, while new business takes the uncapped demand premium. The credibility weight shows how much of each policy's risk cost comes from its own claims history.

In [ ]:
result["policy"][
    [
        "policy.policy_number",
        "policy.transaction_type",
        "policy.vehicle_count",
        "policy.claim_count",
        "policy.credibility_weighting",
        "policy.pred_claim_cost",
        "policy.claims_rated_pred_claim_cost",
        "policy.premium_stat_prem_risk_cost",
        "policy.previous_gwp",
        "policy.gwp",
    ]
].set_index("policy.policy_number").T.round(3)

### Reporting KPIs

In [ ]:
reporting_columns = ["risk.risk_number"] + [layer.output_column for layer in reporting]

result["risk"][reporting_columns].set_index("risk.risk_number").round(3)

## 8. Save using cloudpickle

In [ ]:
import cloudpickle

PIPELINE_PATH = MODELS_DIR / 'rating_engines'/ "pricing_pipeline.pkl"

with PIPELINE_PATH.open("wb") as file:
    cloudpickle.dump(pipeline, file)

print(f"Pipeline saved to {PIPELINE_PATH}")

## 9. Export for the Rust scorer

The pipeline can also be compiled into a **scoring plan**: a JSON description of every layer, in the order the pipeline runs them, with each expression stored as a small tree. `ors-scorer`, a Rust program in `open-rating-engine/scorer`, scores any plan without Python, pandas or a JVM. One quote takes well under a millisecond, and the serving image is a single binary plus the plan.

`export_plan` never raises. It returns the plan, or every layer the plan can't express and why: an H2O model, a `ColumnCalculation` written as a Python function, a `CustomLayer`. This pipeline has none, because both models are ONNX and every calculation is an expression string.

`write_plan_bundle` writes the plan and copies the ONNX files next to it, with their checksums:

```
rust_bundle/
  plan.json
  models/motor_risk_cost_onnx.onnx
  models/motor_demand_onnx.onnx
```

In [ ]:
import json
from collections import Counter

from open_rating_engine.scoring_plan import check_parity, export_plan, write_plan_bundle

export = export_plan(pipeline)
assert export.exportable, "\n".join(str(blocker) for blocker in export.blockers)

BUNDLE_DIR = MODELS_DIR / "rating_engines" / "rust_bundle"
plan_path = write_plan_bundle(pipeline, BUNDLE_DIR)

print(f"Plan written to {plan_path} ({plan_path.stat().st_size / 1024:.0f} KB)")
print(f"{len(export.plan['steps'])} steps:", dict(Counter(step["op"] for step in export.plan["steps"])))

Each step mirrors one layer. The cap/cup calculation, for example, is stored like this. The catalog lookups in the expense layers are evaluated once here and stored as numbers.

In [ ]:
step = next(s for s in export.plan["steps"] if s["layer"] == "risk.premium_comm_prem_cap_cup")
print(json.dumps(step, indent=1)[:1200])

### Get the scorer

The next cell uses a local build of the scorer if there is one (`cargo build --release` in `open-rating-engine/scorer`). Otherwise it uses Docker and builds the `ors-scorer` image from `open-rating-engine/scorer/Dockerfile` the first time, which takes a few minutes.

In [ ]:
import subprocess
import sys
import tempfile
import time

import requests

SCORER_DIR = Path("..").resolve() / "scorer"
SCORER_IMAGE = "ors-scorer:dev"
SERVING_PORT = 5001
LOG_DIR = Path(tempfile.gettempdir()) / "open_rating_engine_scorer"
LOG_DIR.mkdir(exist_ok=True)

binary = SCORER_DIR / "target" / "release" / ("ors-scorer.exe" if sys.platform == "win32" else "ors-scorer")
if binary.exists():
    # check_parity appends "score --bundle <dir>"; serving runs "serve".
    scorer_command = [str(binary)]
    serve_command = [str(binary), "serve", "--bundle", str(BUNDLE_DIR), "--port", str(SERVING_PORT)]
    print(f"Using the local build: {binary}")
else:
    have_image = subprocess.run(["docker", "image", "inspect", SCORER_IMAGE], capture_output=True).returncode == 0
    if not have_image:
        print(f"Building {SCORER_IMAGE} from {SCORER_DIR} ...")
        subprocess.run(["docker", "build", "-t", SCORER_IMAGE, str(SCORER_DIR)], check=True)
    scorer_command = ["docker", "run", "--rm", "-i", "-v", "{bundle}:/bundle", SCORER_IMAGE]
    serve_command = [
        "docker", "run", "--rm", "--name", "ors-scorer-example",
        "-p", f"{SERVING_PORT}:5001", "-v", f"{BUNDLE_DIR}:/app/model", SCORER_IMAGE,
    ]
    print(f"Using the Docker image {SCORER_IMAGE}")

### Check it matches pandas

`check_parity` scores the same inputs with `pipeline.predict` and with the scorer, both as JSON records, then compares every table, column and row. A deploy should only use the Rust scorer for a pipeline that passes this check on its samples. Here it runs on the whole book, so it takes about a minute.

ONNX Runtime adds up tree ensembles in an order that depends on its thread count, so on a large batch a model's float32 outputs change with the number of cores. The scorer runs ONNX on one thread for that reason, and `check_parity` scores the pandas side on one thread too.

What's left are a handful of `risk.conversion_rate` values that differ in the last float32 digit (a relative difference below 1e-6), because this Python environment's ONNX Runtime is a newer version than the one built into the scorer. Everything else matches exactly, so the check allows a relative difference of 1e-5.

In [ ]:
report = check_parity(pipeline, BUNDLE_DIR, data, scorer_command, rtol=1e-5)
print(report.summary())
assert report.ok

### Serve the bundle

`ors-scorer serve` exposes the same endpoints as an MLflow model server: `GET /ping` for health checks and `POST /invocations` for scoring. It loads the plan and models once at start-up, checking the models against the plan's checksums.

In [ ]:
def start(cmd, log_name, health_url, timeout=60):
    """Start a background process, logging to LOG_DIR, and wait until health_url responds."""
    log_path = LOG_DIR / log_name
    log_file = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"Process exited early. Last log lines:\n{log_path.read_text()[-3000:]}")
        try:
            requests.get(health_url, timeout=2).raise_for_status()
            print(f"Ready: {health_url} (log: {log_path})")
            return proc
        except requests.RequestException:
            time.sleep(0.2)
    stop(proc)
    raise TimeoutError(f"{health_url} did not respond within {timeout}s. See {log_path}")


def stop(proc):
    """Stop the scorer, and its container when it runs in Docker."""
    if "docker" in serve_command[0]:
        subprocess.run(["docker", "rm", "-f", "ors-scorer-example"], capture_output=True)
    if proc.poll() is None:
        proc.terminate()
        proc.wait(timeout=30)


serving_proc = start(serve_command, "scorer.log", f"http://127.0.0.1:{SERVING_PORT}/ping")

### Score over REST

The request body uses MLflow's `inputs` format, with one list of records for each input table. `to_json` converts missing values such as the new-business `previous_gwp` to `null`. The premiums should match the local scoring in section 7.

Section 7 ran ONNX on every core, so its model outputs can differ from the scorer's in the last float32 digit (see the parity check above). On the smallest conversion rates that is a relative difference of a few in 100,000, hence `rtol=1e-4` for the model output.

In [ ]:
payload = {"inputs": {table: json.loads(df.to_json(orient="records")) for table, df in data.items()}}

response = requests.post(f"http://127.0.0.1:{SERVING_PORT}/invocations", json=payload, timeout=120)
response.raise_for_status()
served = {table: pd.DataFrame(records) for table, records in response.json()["predictions"].items()}

assert np.allclose(served["policy"]["policy.gwp"], result["policy"]["policy.gwp"])
assert np.allclose(served["risk"]["risk.conversion_rate"], result["risk"]["risk.conversion_rate"], rtol=1e-4)

served["policy"][["policy.policy_number", "policy.transaction_type", "policy.gwp"]]

### One quote

A serving endpoint usually prices one policy at a time. The scorer logs one JSON line per request, with the time it spent scoring (`latency_ms`) and the plan that produced the price (`plan_hash`). The round trip measured here also includes HTTP, and Docker's port forwarding when the scorer runs in a container.

In [ ]:
quote_policy = data["policy"]["policy.policy_number"].iloc[0]
quote = {
    "inputs": {
        table: json.loads(df[df[f"{table}.policy_number"] == quote_policy].to_json(orient="records"))
        for table, df in data.items()
    }
}

round_trips = []
for _ in range(100):
    start_time = time.perf_counter()
    requests.post(f"http://127.0.0.1:{SERVING_PORT}/invocations", json=quote, timeout=10).raise_for_status()
    round_trips.append((time.perf_counter() - start_time) * 1000)

log_lines = [json.loads(line) for line in (LOG_DIR / "scorer.log").read_text().splitlines() if line.startswith("{")]
scoring_ms = [line["latency_ms"] for line in log_lines[-100:]]

print(f"Scoring time in the scorer: median {np.median(scoring_ms):.2f} ms")
print(f"Round trip from this notebook: median {np.median(round_trips):.2f} ms")
print("Last log line:", json.dumps(log_lines[-1]))

### Stop the server

In [ ]:
stop(serving_proc)

### Deploying the scorer

The cells above run the scorer on `127.0.0.1`. A real endpoint follows the same steps, in a pipeline instead of a notebook.

**1. Build the base image once per scorer version**

```bash
docker build -t <registry>/ors-scorer:0.1.0 open-rating-engine/scorer
```

The image holds only the `ors-scorer` binary, with ONNX Runtime linked in. It doesn't change when the rating algorithm changes.

**2. Build one image per algorithm version**

Export the bundle, run `check_parity` on the algorithm's samples, and only then copy the bundle onto the base image. No compiler or `pip install` runs at this step, so it takes seconds:

```dockerfile
FROM <registry>/ors-scorer:0.1.0
COPY rust_bundle /app/model
```

Keep the bundle with the algorithm version it came from, for example as an artifact of the registered model, so every deployed price can be traced to its plan.

**3. Run it like any other API**

- Run the image on a container platform such as Azure Container Apps, Kubernetes or ECS. It listens on port 5001 (`PORT` changes that) and needs a fraction of the memory of the Python image.
- Use `/ping` as the health check. Start-up takes well under a second, which makes scaling to zero practical.
- Put authentication in front of `/invocations`, and run more than one replica.
- Ship the per-request log lines to your log platform. Each one names the `scorer_version` and `plan_hash` that produced the price.

**4. Know what the scorer doesn't do**

- **Input validation:** the scorer refuses to start if the bundle contains an `input_validation.json`, because it doesn't implement those checks. Validate requests before they reach it, or keep the Python image for algorithms that rely on validation.
- **Threads:** ONNX runs on one thread (`ORS_ONNX_THREADS=1`) so a price is the same on every machine. Raise it only for large batch scoring, where the last float32 digit may then vary.
- **Unsupported layers:** an algorithm with a blocker from `export_plan` keeps the Python image.